# 🛣️ AI-Based Road Maintenance Priority Optimizer

MLP road-benefit ranking + exact budget-constrained portfolio optimization.

> Educational planning simulation; benefit weights are declared policy assumptions.

👉 **Open the interactive companion:** [https://road-maintenance-priority.streamlit.app](https://road-maintenance-priority.streamlit.app/?stage=start)

## Complete workflow

Road data → transparent benefit labels → normalized MLP → predicted scores → 0/1 knapsack → repair/defer schedule → policy comparison.

## Interactive learning journey

- [More Damaged Roads Than Budget](https://road-maintenance-priority.streamlit.app/?stage=problem) — Resource-Constrained Decision
- [Five Road Planning Attributes](https://road-maintenance-priority.streamlit.app/?stage=inputs) — Ranking Features
- [What Counts as Maintenance Benefit?](https://road-maintenance-priority.streamlit.app/?stage=benefit) — Supervised Ranking Target
- [A Municipal Road Candidate List](https://road-maintenance-priority.streamlit.app/?stage=data) — Synthetic Training Dataset
- [Putting Road Records on One Scale](https://road-maintenance-priority.streamlit.app/?stage=prepare) — Normalization
- [Estimating Maintenance Priority](https://road-maintenance-priority.streamlit.app/?stage=ranking) — MLP Ranking Model
- [Checking the Priority Scores](https://road-maintenance-priority.streamlit.app/?stage=rank_audit) — Ranking Evaluation
- [Choosing the Affordable Portfolio](https://road-maintenance-priority.streamlit.app/?stage=budget) — 0/1 Knapsack Optimization
- [The Maintenance Programme Audit](https://road-maintenance-priority.streamlit.app/?stage=compare) — Policy Comparison and Governance

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error,mean_squared_error
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input,Dense,Dropout
from tensorflow.keras.callbacks import EarlyStopping
SEED=42;np.random.seed(SEED);tf.random.set_seed(SEED)
FEATURES=["condition_score","traffic_vpd","repair_cost_lakh","accident_risk","importance"]

---
# 1. More Damaged Roads Than Budget
### Phase 1 of 6 · The Budget Dilemma

## Part 1 · In road maintenance
A city maintains many damaged roads but cannot fund every treatment in the current programme.

## Part 2 · The engineering challenge
Repairing only the worst pavement can consume the budget while leaving high-traffic, safety-critical links untreated.

## Part 3 · Where the AI comes in
Estimate each road's public benefit and select the combination with the greatest total benefit under the budget.

**Civil Engineering:** More Damaged Roads Than Budget → **AI:** Resource-Constrained Decision → `which repairs maximize network benefit?`

> 🎬 **See this illustrated and interactive:** [https://road-maintenance-priority.streamlit.app/?stage=problem](https://road-maintenance-priority.streamlit.app/?stage=problem)

## Part 4 · The technical explanation

The neural model estimates individual maintenance benefit. The budget optimizer then chooses a combination. Keeping these tasks separate makes affordability and policy assumptions visible.

## Part 5 · What you just built

**In the notebook:** Define ranking and portfolio selection as separate tasks.

**Takeaway:** The best individual road and the best affordable programme are different questions.

[Project overview](https://road-maintenance-priority.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Five Road Planning Attributes](https://road-maintenance-priority.streamlit.app/?stage=inputs) ▶

---
# 2. Five Road Planning Attributes
### Phase 1 of 6 · The Budget Dilemma

## Part 1 · In road maintenance
Each candidate has a condition score, daily traffic, repair cost, accident risk, and strategic importance.

## Part 2 · The engineering challenge
The inputs use different units, and cost affects affordability while the other attributes largely describe benefit.

## Part 3 · Where the AI comes in
Keep five named inputs, but audit whether cost should influence ranking or only the downstream budget optimizer.

**Civil Engineering:** Five Road Planning Attributes → **AI:** Ranking Features → `condition, traffic, cost, accident risk, importance`

> 🎬 **See this illustrated and interactive:** [https://road-maintenance-priority.streamlit.app/?stage=inputs](https://road-maintenance-priority.streamlit.app/?stage=inputs)

## Part 4 · The technical explanation

In [ ]:
example=dict(condition_score=35,traffic_vpd=18000,repair_cost_lakh=12,accident_risk=8,importance=9)
pd.Series(example,name="Road A")

## Part 5 · What you just built

**In the notebook:** Create the road table and inspect physical/planning ranges.

**Takeaway:** Feature roles must be defined before training: benefit evidence is not identical to budget cost.

◀ [Previous: More Damaged Roads Than Budget](https://road-maintenance-priority.streamlit.app/?stage=problem) &nbsp;|&nbsp; [Project overview](https://road-maintenance-priority.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: What Counts as Maintenance Benefit?](https://road-maintenance-priority.streamlit.app/?stage=benefit) ▶

---
# 3. What Counts as Maintenance Benefit?
### Phase 2 of 6 · Defining Public Benefit

## Part 1 · In road maintenance
A planning policy must state how pavement distress, usage, safety, and critical connections contribute to benefit.

## Part 2 · The engineering challenge
Hidden or arbitrary weights can turn an AI score into an unreviewable policy decision.

## Part 3 · Where the AI comes in
Generate a transparent educational benefit target with declared weights and noise, then let the MLP learn its combined pattern.

**Civil Engineering:** What Counts as Maintenance Benefit? → **AI:** Supervised Ranking Target → `weighted damage + traffic + risk + importance`

> 🎬 **See this illustrated and interactive:** [https://road-maintenance-priority.streamlit.app/?stage=benefit](https://road-maintenance-priority.streamlit.app/?stage=benefit)

## Part 4 · The technical explanation

In [ ]:
def policy_benefit(df):
 damage=(100-df.condition_score)/80;traffic=df.traffic_vpd/30000;risk=df.accident_risk/10;importance=df.importance/10
 return 100*(.35*damage+.25*traffic+.22*risk+.18*importance)
print("Declared weights: damage 35%, traffic 25%, accident risk 22%, importance 18%")

## Part 5 · What you just built

**In the notebook:** Calculate normalized component benefits and the training target.

**Takeaway:** The model learns the policy encoded in its labels; it does not discover public values independently.

◀ [Previous: Five Road Planning Attributes](https://road-maintenance-priority.streamlit.app/?stage=inputs) &nbsp;|&nbsp; [Project overview](https://road-maintenance-priority.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: A Municipal Road Candidate List](https://road-maintenance-priority.streamlit.app/?stage=data) ▶

---
# 4. A Municipal Road Candidate List
### Phase 2 of 6 · Defining Public Benefit

## Part 1 · In road maintenance
Road condition, traffic, risk, importance, and treatment cost vary across a municipal network.

## Part 2 · The engineering challenge
Random independent columns miss realistic relationships such as more extensive damage generally costing more to repair.

## Part 3 · Where the AI comes in
Generate overlapping, correlated road records and reserve a separate current-year programme for optimization.

**Civil Engineering:** A Municipal Road Candidate List → **AI:** Synthetic Training Dataset → `hundreds of roads and plausible correlations`

> 🎬 **See this illustrated and interactive:** [https://road-maintenance-priority.streamlit.app/?stage=data](https://road-maintenance-priority.streamlit.app/?stage=data)

## Part 4 · The technical explanation

In [ ]:
rng=np.random.default_rng(SEED);n=5000;condition=rng.uniform(18,88,n);traffic=np.clip(rng.lognormal(9.2,.55,n),1200,32000);risk=np.clip((100-condition)/12+rng.normal(2,1.5,n),1,10);importance=rng.uniform(1,10,n);cost=np.clip((100-condition)*.16+traffic/9000+rng.normal(2,2,n),3,28)
data=pd.DataFrame(dict(condition_score=condition,traffic_vpd=traffic,repair_cost_lakh=cost,accident_risk=risk,importance=importance));data["benefit"]=policy_benefit(data)+rng.normal(0,3,n)
print(data.describe().T);data.head()

## Part 5 · What you just built

**In the notebook:** Build training roads and a named 20-road planning list.

**Takeaway:** Synthetic data demonstrates method, not a real authority's priorities.

◀ [Previous: What Counts as Maintenance Benefit?](https://road-maintenance-priority.streamlit.app/?stage=benefit) &nbsp;|&nbsp; [Project overview](https://road-maintenance-priority.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Putting Road Records on One Scale](https://road-maintenance-priority.streamlit.app/?stage=prepare) ▶

---
# 5. Putting Road Records on One Scale
### Phase 3 of 6 · Learning Priority

## Part 1 · In road maintenance
Traffic may be tens of thousands of vehicles while risk and importance use small ordinal scales.

## Part 2 · The engineering challenge
Raw magnitude distorts MLP training, and full-dataset scaling leaks test information.

## Part 3 · Where the AI comes in
Split first, then learn imputation and scaling parameters from training records only.

**Civil Engineering:** Putting Road Records on One Scale → **AI:** Normalization → `training-only imputer and StandardScaler`

> 🎬 **See this illustrated and interactive:** [https://road-maintenance-priority.streamlit.app/?stage=prepare](https://road-maintenance-priority.streamlit.app/?stage=prepare)

## Part 4 · The technical explanation

In [ ]:
train,temp=train_test_split(data,test_size=.30,random_state=SEED);val,test=train_test_split(temp,test_size=.50,random_state=SEED);imputer=SimpleImputer(strategy="median").fit(train[FEATURES]);scaler=StandardScaler().fit(imputer.transform(train[FEATURES]));prep=lambda d:scaler.transform(imputer.transform(d[FEATURES]));Xtr,Xva,Xte=prep(train),prep(val),prep(test);ytr,yva,yte=train.benefit.to_numpy(),val.benefit.to_numpy(),test.benefit.to_numpy();print(Xtr.shape,Xva.shape,Xte.shape)

## Part 5 · What you just built

**In the notebook:** Prepare five features without leakage.

**Takeaway:** Preprocessing is part of the ranking model and must be reproduced during planning.

◀ [Previous: A Municipal Road Candidate List](https://road-maintenance-priority.streamlit.app/?stage=data) &nbsp;|&nbsp; [Project overview](https://road-maintenance-priority.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Estimating Maintenance Priority](https://road-maintenance-priority.streamlit.app/?stage=ranking) ▶

---
# 6. Estimating Maintenance Priority
### Phase 3 of 6 · Learning Priority

## Part 1 · In road maintenance
Engineers need comparable road-level benefit estimates before assembling a programme.

## Part 2 · The engineering challenge
A score order alone ignores budget interactions and can favour expensive roads that crowd out a better combination.

## Part 3 · Where the AI comes in
Train a compact MLP to estimate priority benefit, then pass predictions—not ranks—to the optimizer.

**Civil Engineering:** Estimating Maintenance Priority → **AI:** MLP Ranking Model → `5 -> Dense32 -> Dense16 -> priority score`

> 🎬 **See this illustrated and interactive:** [https://road-maintenance-priority.streamlit.app/?stage=ranking](https://road-maintenance-priority.streamlit.app/?stage=ranking)

## Part 4 · The technical explanation

In [ ]:
model=Sequential([Input((5,)),Dense(32,activation="relu"),Dropout(.1),Dense(16,activation="relu"),Dense(1)]);model.compile(optimizer="adam",loss="mae");early=EarlyStopping(monitor="val_loss",patience=7,restore_best_weights=True);model.fit(Xtr,ytr,validation_data=(Xva,yva),epochs=60,batch_size=64,callbacks=[early],verbose=0);model.summary()

## Part 5 · What you just built

**In the notebook:** Train the regression MLP and score the current programme roads.

**Takeaway:** Ranking estimates individual value; optimization decides collective selection.

◀ [Previous: Putting Road Records on One Scale](https://road-maintenance-priority.streamlit.app/?stage=prepare) &nbsp;|&nbsp; [Project overview](https://road-maintenance-priority.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Checking the Priority Scores](https://road-maintenance-priority.streamlit.app/?stage=rank_audit) ▶

---
# 7. Checking the Priority Scores
### Phase 3 of 6 · Learning Priority

## Part 1 · In road maintenance
Planning quality depends on preserving the relative order of high-benefit roads, not only predicting exact score values.

## Part 2 · The engineering challenge
Low MAE can coexist with poor ordering near the funding cutoff.

## Part 3 · Where the AI comes in
Audit numeric error, rank correlation, and top-k overlap, especially for safety-critical roads.

**Civil Engineering:** Checking the Priority Scores → **AI:** Ranking Evaluation → `MAE, Spearman correlation, top-k overlap`

> 🎬 **See this illustrated and interactive:** [https://road-maintenance-priority.streamlit.app/?stage=rank_audit](https://road-maintenance-priority.streamlit.app/?stage=rank_audit)

## Part 4 · The technical explanation

In [ ]:
pred=model.predict(Xte,verbose=0).ravel();print("MAE:",mean_absolute_error(yte,pred));print("RMSE:",mean_squared_error(yte,pred)**.5);print("Spearman rank correlation:",spearmanr(yte,pred).statistic)
k=50;overlap=len(set(np.argsort(yte)[-k:])&set(np.argsort(pred)[-k:]));print(f"Top-{k} overlap:",overlap,"of",k);plt.scatter(yte,pred,s=8,alpha=.3);plt.xlabel("Target benefit");plt.ylabel("Predicted benefit");plt.grid(alpha=.2);plt.show()

## Part 5 · What you just built

**In the notebook:** Evaluate unseen-road scores and inspect large ranking disagreements.

**Takeaway:** A ranking model should be evaluated as a ranking model.

◀ [Previous: Estimating Maintenance Priority](https://road-maintenance-priority.streamlit.app/?stage=ranking) &nbsp;|&nbsp; [Project overview](https://road-maintenance-priority.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Choosing the Affordable Portfolio](https://road-maintenance-priority.streamlit.app/?stage=budget) ▶

---
# 8. Choosing the Affordable Portfolio
### Phase 4 of 6 · Selecting the Programme

## Part 1 · In road maintenance
Each road is either funded this programme or deferred, and the total treatment cost cannot exceed the approved budget.

## Part 2 · The engineering challenge
Selecting in score order can leave unusable budget or miss a combination with greater total benefit.

## Part 3 · Where the AI comes in
Use exact dynamic programming at ₹1-lakh resolution to find the highest predicted-benefit combination.

**Civil Engineering:** Choosing the Affordable Portfolio → **AI:** 0/1 Knapsack Optimization → `maximize predicted benefit subject to total cost ≤ budget`

> 🎬 **See this illustrated and interactive:** [https://road-maintenance-priority.streamlit.app/?stage=budget](https://road-maintenance-priority.streamlit.app/?stage=budget)

## Part 4 · The technical explanation

In [ ]:
programme=data.sample(20,random_state=9).reset_index(drop=True);programme["Road"]=[f"Road {chr(65+i)}" for i in range(20)];programme["predicted_benefit"]=model.predict(prep(programme),verbose=0).ravel();programme["cost_int"]=programme.repair_cost_lakh.round().clip(1).astype(int)
def knapsack(df,budget):
 n=len(df);dp=np.zeros((n+1,budget+1));take=np.zeros((n+1,budget+1),bool)
 for i in range(1,n+1):
  c=int(df.iloc[i-1].cost_int);v=float(df.iloc[i-1].predicted_benefit);dp[i]=dp[i-1]
  for b in range(c,budget+1):
   if dp[i-1,b-c]+v>dp[i,b]:dp[i,b]=dp[i-1,b-c]+v;take[i,b]=True
 b=budget;sel=[]
 for i in range(n,0,-1):
  if take[i,b]:sel.append(i-1);b-=int(df.iloc[i-1].cost_int)
 return sorted(sel),dp[n,budget]
selected,total=knapsack(programme,50);programme["Decision"]=["REPAIR" if i in selected else "DEFER" for i in range(len(programme))];display(programme[["Road","predicted_benefit","cost_int","Decision"]].sort_values("predicted_benefit",ascending=False));print("Budget used:",programme.iloc[selected].cost_int.sum(),"lakh | Total predicted benefit:",total)

## Part 5 · What you just built

**In the notebook:** Run knapsack and produce selected/deferred lists and budget use.

**Takeaway:** The optimizer chooses combinations; a sorted list does not.

◀ [Previous: Checking the Priority Scores](https://road-maintenance-priority.streamlit.app/?stage=rank_audit) &nbsp;|&nbsp; [Project overview](https://road-maintenance-priority.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: The Maintenance Programme Audit](https://road-maintenance-priority.streamlit.app/?stage=compare) ▶

---
# 9. The Maintenance Programme Audit
### Phase 5 of 6 · Comparing Policies

## Part 1 · In road maintenance
Decision-makers need to see what improved, who was deferred, and how sensitive the plan is to budget and policy weights.

## Part 2 · The engineering challenge
A numerically optimal programme can still conflict with statutory duties, geographic equity, emergency routes, or minimum service obligations.

## Part 3 · Where the AI comes in
Compare policies, vary budget, report uncertainty, and keep final approval with accountable planners.

**Civil Engineering:** The Maintenance Programme Audit → **AI:** Policy Comparison and Governance → `worst-first vs score-first vs optimized`

> 🎬 **See this illustrated and interactive:** [https://road-maintenance-priority.streamlit.app/?stage=compare](https://road-maintenance-priority.streamlit.app/?stage=compare)

## Part 4 · The technical explanation

In [ ]:
def greedy(df,budget,order):
 used=0;sel=[]
 for i in order:
  c=int(df.iloc[i].cost_int)
  if used+c<=budget:sel.append(i);used+=c
 return sel
worst=greedy(programme,50,np.argsort(programme.condition_score));score=greedy(programme,50,np.argsort(programme.predicted_benefit)[::-1]);opt,_=knapsack(programme,50)
rows=[]
for name,sel in [("Worst condition first",worst),("Highest score first",score),("Knapsack optimized",opt)]:rows.append(dict(Policy=name,Roads=len(sel),Cost=int(programme.iloc[sel].cost_int.sum()),Benefit=float(programme.iloc[sel].predicted_benefit.sum())))
display(pd.DataFrame(rows));budgets=np.arange(10,101,5);values=[knapsack(programme,int(b))[1] for b in budgets];plt.plot(budgets,values,"o-");plt.xlabel("Budget ₹ lakh");plt.ylabel("Maximum predicted benefit");plt.grid(alpha=.2);plt.show()
print("Governance limits: weights are assumptions; no lifecycle treatment choice, deterioration forecast, geographic equity, accessibility duty, utility coordination, inflation, procurement, or uncertainty optimization.")

## Part 5 · What you just built

**In the notebook:** Show schedule, cost, benefit, sensitivity, and limitations.

**Takeaway:** Optimization supports a transparent programme; it does not replace public-sector governance.

◀ [Previous: Choosing the Affordable Portfolio](https://road-maintenance-priority.streamlit.app/?stage=budget) &nbsp;|&nbsp; [Project overview](https://road-maintenance-priority.streamlit.app/?stage=start)

---
# Final engineering conclusion

The MLP estimates road-level maintenance benefit from five attributes. Exact knapsack optimization then selects the highest-benefit affordable combination. Separating score prediction from portfolio selection keeps policy, cost, and budget constraints reviewable.